In [1]:
"""
🧹 Data Lakehouse: Silver Zone (Cleaning Zone) Pipeline & QA Dashboard
Architecture: 4-Zone Data Lakehouse
Engine: Functional Monad-based Data Quality Engine (v1.3)
"""

import os
import datetime
import warnings
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

# Suppress unnecessary warnings for cleaner console output
warnings.filterwarnings('ignore')

try:
    import altair as alt
    # Disable max rows for Altair to handle large datasets in logs
    alt.data_transformers.disable_max_rows()
except ImportError:
    print('⚠️ Warning: Altair library not found. Dashboard generation will be skipped.')
    alt = None

# ==========================================
# Pipeline Paths Configuration
# ==========================================
# ปรับ Path ให้ตรงกับ Volume ที่ Mount ใน Docker
RAW_DATA_DIR = "../data/rawData/"
CLEAN_DATA_DIR = "../data/clean_zone/"
REPORT_DIR = "../data/reports/"

CHINOOK_DB = os.path.join(RAW_DATA_DIR, "chinook.db")
NORTHWIND_DB = os.path.join(RAW_DATA_DIR, "northwind.db")

# Set execution run date partition (YYYY-MM-DD)
RUN_DATE = datetime.datetime.now().strftime('%Y-%m-%d')

# Ensure directories exist
os.makedirs(CLEAN_DATA_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print(f"✅ Configuration Initialized.")
print(f"   • Raw Data Directory  : {RAW_DATA_DIR}")
print(f"   • Clean Data Directory: {CLEAN_DATA_DIR}")
print(f"   • Execution Partition : run_date={RUN_DATE}")
print("="*60)


# ==========================================
# Monad-based Data Quality Engine
# ==========================================
class Monad:
    """
    Functional Data Quality Monad for state management and continuous validation logging.
    """
    __slots__ = ['value', 'input_row', 'output_column', 'status', 'final_value', 'logs', 'stopped', 'step']

    def __init__(self, value, row, col):
        self.value = value
        self.input_row = row
        self.output_column = col
        self.status = 'pending'
        self.final_value = None
        self.logs = []
        self.stopped = False
        self.step = 0

    def _log(self, func, status, details=None):
        role = getattr(func, '_role', 'unknown')
        name = getattr(func, '__name__', 'unknown')
        self.logs.append({
            'row': self.input_row,
            'col': self.output_column,
            'step': self.step,
            'role_type': role,
            'rule_name': name,
            'status': status,
            'value': str(self.value),
            'details': details
        })

    def __or__(self, func):
        if self.stopped:
            self.step += 1
            self._log(func, 'skipped')
            return self
            
        self.step += 1
        role = getattr(func, '_role', 'validator')
        try:
            res = func(self.value)
            if role == 'validator':
                self.value = res
                self.status = 'valid'
                self._log(func, 'passed')
            elif role == 'transformer':
                self.value = res
                self.status = 'success'
                self.final_value = res
                self.stopped = True 
                self._log(func, 'passed')
            elif role == 'fallback' and self.status in ['dirty', 'error']:
                self.value = res
                self.status = 'recovered'
                self.final_value = res
                self.stopped = True
                self._log(func, 'recovered')
        except Exception as e:
            if role == 'validator':
                self.status = 'dirty'
                self.final_value = None
                self.stopped = True
                self._log(func, 'failed', str(e))
            elif role == 'transformer':
                self.status = 'error'
                self.final_value = None
                self.stopped = True
                self._log(func, 'failed', str(e))
        return self


# ==========================================
# Monad Rules & Decorators
# ==========================================
def validator(func):
    func._role = 'validator'
    return func

def transformer(func):
    func._role = 'transformer'
    return func

@validator
def not_null(val):
    """Ensure value is not null, NaN, or empty string."""
    if pd.isna(val) or val is None or str(val).strip() == '':
        raise ValueError("Value cannot be null or empty")
    return val

@validator
def is_positive(val):
    """Ensure numerical value is non-negative."""
    if float(val) < 0:
        raise ValueError("Value must be non-negative")
    return val

@transformer
def fill_unknown(val):
    """Fallback transformer: Fill missing values with 'UNKNOWN'."""
    if pd.isna(val) or val is None or str(val).strip() == '':
        return "UNKNOWN"
    raise ValueError("Value is already valid; skipping transformation")

@transformer
def fill_zero(val):
    """Fallback transformer: Fill missing values with 0."""
    if pd.isna(val) or val is None:
        return 0
    raise ValueError("Value is already valid; skipping transformation")


# ==========================================
# Metadata & Schema Specifications
# ==========================================
SCHEMA_CONFIG = {
    "chinook": {
        "Customer": {
            "CustomerId": [not_null], "FirstName": [not_null], "LastName": [not_null], "Company": [fill_unknown]
        },
        "Employee": {
            "EmployeeId": [not_null], "FirstName": [not_null], "LastName": [not_null], "Title": [fill_unknown]
        },
        "Invoice": {
            "InvoiceId": [not_null], "CustomerId": [not_null], "Total": [not_null, is_positive]
        },
        "InvoiceLine": {
            "InvoiceLineId": [not_null], "InvoiceId": [not_null], "TrackId": [not_null], "UnitPrice": [not_null, is_positive]
        },
        "Track": {
            "TrackId": [not_null], "Name": [not_null], "Milliseconds": [is_positive]
        },
        "Album": {
            "AlbumId": [not_null], "Title": [not_null], "ArtistId": [not_null]
        }
    },
    "northwind": {
        "Customers": {
            "CustomerID": [not_null], "CompanyName": [not_null], "ContactName": [fill_unknown]
        },
        "Employees": {
            "EmployeeID": [not_null], "LastName": [not_null], "FirstName": [not_null]
        },
        "Orders": {
            "OrderID": [not_null], "CustomerID": [not_null], "EmployeeID": [not_null]
        },
        "Order Details": {
            "OrderID": [not_null], "ProductID": [not_null], "UnitPrice": [not_null, is_positive], "Quantity": [not_null, is_positive]
        },
        "Products": {
            "ProductID": [not_null], "ProductName": [not_null], "UnitPrice": [is_positive]
        }
    }
}


def process_dataframe(df: pd.DataFrame, schema: dict, table_name: str, db_name: str):
    """
    Applies Data Quality rules column-by-column using Monad, collecting detailed audit logs.
    """
    all_logs = []
    df_clean = df.copy()
    
    # Process only columns defined in the schema for the specific table
    for col, rules in schema.items():
        if col not in df_clean.columns:
            continue
            
        cleaned_col = []
        for idx, val in enumerate(df_clean[col]):
            m = Monad(val, idx, col)
            for rule in rules:
                m = m | rule
            
            # Enrich audit logs with metadata
            for log in m.logs:
                log['db_name'] = db_name
                log['table_name'] = table_name
                
            all_logs.extend(m.logs)
            
            # Derive cleaned value
            final_val = m.final_value if m.final_value is not None else m.value
            cleaned_col.append(final_val)
            
        df_clean[col] = cleaned_col
        
    return df_clean, all_logs


# ==========================================
# Pipeline Execution Main Loop
# ==========================================
def main():
    master_logs = []
    dbs = {
        "chinook": CHINOOK_DB,
        "northwind": NORTHWIND_DB
    }

    print(f"🚀 Starting Data Pipeline Ingestion for Run Date: {RUN_DATE}")
    
    for db_name, db_path in dbs.items():
        if not os.path.exists(db_path):
            print(f"❌ Error: Database file not found -> {db_path}")
            continue
            
        print(f"\n📂 Connecting to {db_name.upper()} database...")
        conn = duckdb.connect(db_path)
        
        db_schema = SCHEMA_CONFIG.get(db_name, {})
        
        for table_name, table_schema in db_schema.items():
            print(f"  ⏳ Extracting & Cleaning: {table_name}...")
            
            try:
                # Extract Data via DuckDB
                # Note: 'Order Details' requires quotes in SQL
                safe_table_name = f'"{table_name}"' if ' ' in table_name else table_name
                df = conn.execute(f"SELECT * FROM {safe_table_name}").df()
                
                # Transform (Data Quality Engine)
                df_clean, logs = process_dataframe(df, table_schema, table_name, db_name)
                master_logs.extend(logs)
                
                # Load (Add Partition Column and Save to Parquet)
                df_clean['run_date'] = RUN_DATE
                
                # Create PyArrow Table
                table = pa.Table.from_pandas(df_clean)
                
                # Write to Hive-partitioned directory structure
                # E.g. ../data/clean_zone/chinook/Customer/run_date=2023-10-27/
                dataset_path = os.path.join(CLEAN_DATA_DIR, db_name, table_name.replace(" ", "_"))
                
                pq.write_to_dataset(
                    table,
                    root_path=dataset_path,
                    partition_cols=['run_date'],
                    existing_data_behavior='overwrite_or_ignore'
                )
                
                print(f"     ✅ Saved clean data to: {dataset_path}")
                
            except Exception as e:
                print(f"     ❌ Failed to process {table_name}: {str(e)}")
                
        conn.close()

    # ==========================================
    # Generate QA Dashboard
    # ==========================================
    if master_logs and alt:
        print("\n📊 Generating Data Quality Dashboard...")
        logs_df = pd.DataFrame(master_logs)
        
        # Summary statistics for console
        total_checks = len(logs_df)
        failed_checks = len(logs_df[logs_df['status'] == 'failed'])
        passed_checks = len(logs_df[logs_df['status'] == 'passed'])
        
        print(f"   • Total Rules Applied: {total_checks}")
        print(f"   • Passed             : {passed_checks}")
        print(f"   • Failed             : {failed_checks}")
        
        # Dashboard Creation
        # 1. Bar chart: Rule Status Distribution
        status_chart = alt.Chart(logs_df).mark_bar().encode(
            x=alt.X('status:N', title='Rule Status'),
            y=alt.Y('count():Q', title='Count of Logs'),
            color=alt.Color('status:N', scale=alt.Scale(domain=['passed', 'failed', 'recovered', 'skipped'], 
                                                        range=['#2ca02c', '#d62728', '#ff7f0e', '#7f7f7f']))
        ).properties(
            title='Overall Data Quality Rule Status',
            width=300,
            height=300
        )
        
        # 2. Heatmap: Quality issues by Table and Rule
        failed_logs = logs_df[logs_df['status'] == 'failed']
        if not failed_logs.empty:
            heatmap = alt.Chart(failed_logs).mark_rect().encode(
                x=alt.X('rule_name:N', title='Validation Rule'),
                y=alt.Y('table_name:N', title='Table'),
                color=alt.Color('count():Q', title='Failed Count', scale=alt.Scale(scheme='reds'))
            ).properties(
                title='Heatmap: Failures by Table and Rule',
                width=400,
                height=300
            )
            dashboard = status_chart | heatmap
        else:
            dashboard = status_chart
            
        # Save as HTML
        dashboard_path = os.path.join(REPORT_DIR, f"qa_dashboard_{RUN_DATE}.html")
        dashboard.save(dashboard_path)
        print(f"   ✅ QA Dashboard saved successfully to: {dashboard_path}")
        
    elif not master_logs:
        print("\n⚠️ No logs generated. Pipeline may not have processed any data.")
        
    print("\n🎉 Pipeline Execution Completed Successfully!")

if __name__ == "__main__":
    main()

✅ Configuration Initialized.
   • Raw Data Directory  : ../data/rawData/
   • Clean Data Directory: ../data/clean_zone/
   • Execution Partition : run_date=2026-08-10
🚀 Starting Data Pipeline Ingestion for Run Date: 2026-08-10

📂 Connecting to CHINOOK database...
  ⏳ Extracting & Cleaning: Customer...
     ✅ Saved clean data to: ../data/clean_zone/chinook/Customer
  ⏳ Extracting & Cleaning: Employee...
     ✅ Saved clean data to: ../data/clean_zone/chinook/Employee
  ⏳ Extracting & Cleaning: Invoice...
     ✅ Saved clean data to: ../data/clean_zone/chinook/Invoice
  ⏳ Extracting & Cleaning: InvoiceLine...
     ✅ Saved clean data to: ../data/clean_zone/chinook/InvoiceLine
  ⏳ Extracting & Cleaning: Track...
     ✅ Saved clean data to: ../data/clean_zone/chinook/Track
  ⏳ Extracting & Cleaning: Album...
     ✅ Saved clean data to: ../data/clean_zone/chinook/Album

📂 Connecting to NORTHWIND database...
  ⏳ Extracting & Cleaning: Customers...
     ✅ Saved clean data to: ../data/clean_zone/

: 